# Day 2 · 3교시 · AWQ 생성 (4B & 8B) — 핵심

> **VLM 경량화 2일 과정 · Day 2 (3교시) · 실습**
> 실습 환경: **RunPod A100 80GB** · 커널: **`Python (vlm_quant)`** · 데이터: **HuggingFaceM4/ChartQA**

---

## 이 교시의 목표
- `llm-compressor`의 **`AWQModifier` + `QuantizationModifier`(W4A16)** 로 멀티모달 calibration을 돌린다.
- **비전 타워·`lm_head`를 `ignore`** (LLM 선형층만 4bit).
- **병합 4B · 베이스 8B를 각각 AWQ로 변환**(`oneshot`)하고 저장한다.
- 원본 대비 **디스크 크기**를 비교한다.

> ⚠️ **`Python (vlm_quant)` 커널**인지 확인하세요(Day2-2에서 만든 venv). 아니면 import가 실패합니다.


## 0. 공통 헤더 — RunPod 작업 폴더(VLM_FT_2) + HF_TOKEN 로드

> 📌 **모든 Day 2 노트북은 이 셀을 먼저 실행합니다.** RunPod 영구 볼륨의 작업 폴더 **`/workspace/VLM_FT_2`** 를 기준으로 잡고, `.env`의 **HF_TOKEN**을 불러옵니다. (Day2는 RunPod이라 Google Drive 마운트가 없습니다.)
> - `VLM_DIR` / `DATA_DIR` : 교시 간 공유 폴더(모델·AWQ 결과·평가/벤치 JSON이 여기 모입니다).
> - **HF_TOKEN**: `VLM_FT_2/.env` 에 `HF_TOKEN=hf_...` 한 줄을 넣어두면 자동 로드됩니다. `login()`은 호출하지 않습니다(환경변수만으로 충분).

In [1]:
# ════════════════════════════════════════════════════════════
#  공통 헤더 · RunPod 작업 폴더(VLM_FT_2) + HF_TOKEN(.env) 로드
#  (모든 Day2 노트북 상단에서 동일하게 실행)
# ════════════════════════════════════════════════════════════
import os

# (1) RunPod 영구 볼륨의 작업 폴더 VLM_FT_2 (교시 간 모델·결과 공유). Drive 마운트 불필요.
VLM_DIR = '/workspace/VLM_FT_2'
DATA_DIR = f'{VLM_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# (2) .env 에서 HF_TOKEN 로드. login()은 부르지 않음(환경변수만으로 인증, 경고 없음).
try:
    from dotenv import load_dotenv
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv
ENV_PATH = f'{VLM_DIR}/.env'
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH)
    print('HF_TOKEN:', '로드됨' if os.environ.get('HF_TOKEN') else '없음')
else:
    print(f'.env 없음 — {ENV_PATH} 에 HF_TOKEN=hf_... 한 줄을 만들면 자동 로드됩니다(공개 모델만 쓰면 없어도 됨)')
print('작업 폴더 VLM_DIR =', VLM_DIR)

HF_TOKEN: 로드됨
작업 폴더 VLM_DIR = /workspace/VLM_FT_2


**▸ 실행 결과 읽기.** `HF_TOKEN: 로드됨` 이면 `.env` 인증 성공, 아래 줄에 작업 폴더(`/workspace/VLM_FT_2`)가 찍힙니다. 이 두 줄이면 공통 헤더 준비 완료입니다.

## 0. 준비 — import & 모델 식별자

AWQ에 필요한 클래스를 불러옵니다. **`AWQModifier`의 import 경로가 llmcompressor 버전마다 다릅니다**(최신은 `modifiers.transform.awq`, 구버전은 `modifiers.awq`) → `try/except`로 둘 다 시도해 어느 버전이든 동작하게 합니다.

In [ ]:
# ── import ───────────────────────────────────────────────────
# 표준 라이브러리:
# - base64: 이미지 바이트를 base64 문자열로 변환
# - os: 환경변수/경로 처리
# - gc: 가비지 컬렉션(메모리 정리)
# - json: JSON 입출력
import base64, os, gc, json

# 메모리 상 바이트 버퍼(이미지 저장/변환용)
from io import BytesIO

# PyTorch (텐서, CUDA 사용)
import torch

# Hugging Face 데이터셋 로더 (ChartQA calibration 데이터 로드)
from datasets import load_dataset

# Qwen VL 유틸: 멀티모달 메시지에서 vision 입력 추출
from qwen_vl_utils import process_vision_info

# Qwen3-VL 모델/프로세서 로드
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

# llm-compressor one-shot 압축 실행 함수
from llmcompressor import oneshot

# 실제 양자화(W4A16) 수행 modifier
from llmcompressor.modifiers.quantization import QuantizationModifier

# AWQModifier import 경로 호환 처리
# - 최신 버전: llmcompressor.modifiers.transform.awq
# - 구버전:   llmcompressor.modifiers.awq
try:
    from llmcompressor.modifiers.transform.awq import AWQModifier
except ImportError:
    from llmcompressor.modifiers.awq import AWQModifier

# CUDA 장치 인식 확인 (예: NVIDIA A100 80GB)
print('import OK | cuda:', torch.cuda.get_device_name(0))

import OK | cuda: NVIDIA A100 80GB PCIe


**▸ 실행 결과 읽기.** `import OK | cuda: NVIDIA A100 80GB …` 가 보이면 모든 클래스 로드 성공 + GPU 인식 완료입니다. 여기서 `ImportError`가 나면 커널이 **`Python (vlm_quant)`** 이 아닐 가능성이 큽니다(Day2-2에서 만든 venv 커널을 선택하세요).

In [3]:
# ── HF 토큰 확인 + 모델 식별자 설정 ─────────────────────────────
# 전제:
# - 공통 헤더 셀에서 VLM_DIR, ENV_PATH가 이미 정의되어 있음
# - 공통 헤더 셀에서 .env 로드가 먼저 실행되었으면 HF_TOKEN이 환경변수에 존재함

# 1) HF_TOKEN이 없으면(최초 1회) 사용자에게 입력받아 .env 파일에 저장
if not os.environ.get('HF_TOKEN'):
    from getpass import getpass
    from dotenv import load_dotenv

    # 터미널에 토큰을 안전하게 입력(입력값 비노출)
    tok = getpass('HF 토큰 입력(최초 1회): ').strip()

    # VLM_FT_2/.env 에 HF_TOKEN 한 줄로 저장
    with open(ENV_PATH, 'w') as f:
        f.write(f'HF_TOKEN={tok}\n')

    # 방금 저장한 .env를 다시 로드해서 현재 세션 환경변수에 반영
    load_dotenv(ENV_PATH)
    print('.env 생성:', ENV_PATH)

# 2) 최종 확인: HF_TOKEN이 반드시 있어야 다음 단계 진행 가능
assert os.environ.get('HF_TOKEN'), '.env 에 HF_TOKEN 이 없습니다'
print('HF 토큰 준비 완료 (.env 환경변수)')

# 3) 모델/어댑터 식별자
BASE_4B = 'Qwen/Qwen3-VL-4B-Instruct'                 # 4B 베이스 모델
ADAPTER_REPO = '<your-username>/Qwen3-VL-4B-ChartQA-lora'  # Day1-5에서 업로드한 어댑터
BASE_8B = 'Qwen/Qwen3-VL-8B-Instruct'                 # 8B 베이스 모델

# 4) 실습 공통 설정값
WORKDIR = VLM_DIR      # 공통 헤더에서 만든 작업 폴더(/workspace/VLM_FT_2)
N_CALIB = 256          # AWQ calibration 샘플 수 (시간 부족 시 128로 축소 가능)
MAX_SEQ_LEN = 2048     # calibration 시 최대 시퀀스 길이

HF 토큰 준비 완료 (.env 환경변수)


**▸ 실행 결과 읽기.** `HF 토큰 준비 완료` 면 인증 OK. 이어 정의된 값의 뜻:
- **`N_CALIB = 256`**: AWQ가 "중요 채널"을 찾는 데 쓰는 **보정(calibration) 샘플 수**. 많을수록 통계가 안정적이지만 느려집니다(시간이 빠듯하면 128).
- **`MAX_SEQ_LEN = 2048`**: 보정 시 한 샘플의 최대 토큰 길이.
- 모델 식별자: 4B는 **base + 어댑터(병합)**, 8B는 **base만** 양자화합니다.

## 1. AWQ 양자화 함수 — 4B·8B 공용

### AWQ가 왜/어떻게 동작하나 (직관)
4bit로 그냥 반올림하면 **큰 값을 갖는 중요한 채널**이 뭉개져 정확도가 떨어질 수 있습니다. AWQ(Activation-aware Weight Quantization)의 아이디어는 간단합니다:
1. 보정 데이터를 흘려 **어느 입력 채널이 중요한지(=활성값이 큰지)** 측정하고,
2. 중요한 채널은 **활성 스케일을 낮추고 대응 가중치 스케일을 높여**(곱이 같으니 출력은 동일) 양자화 오차(MSE)를 줄입니다.

즉 "출력을 바꾸지 않으면서 중요한 가중치를 양자화하기 쉽게 만드는" 전처리입니다.

### 이 함수의 레시피 (2-modifier)
| 단계 | 내용 |
|---|---|
| 전처리 | 이미지→base64 → 채팅 템플릿 → `process_vision_info` → `processor(...)` |
| 레시피 | **`AWQModifier()`**(중요 채널 탐색·스무딩) + **`QuantizationModifier(W4A16)`**(실제 4bit 변환) |
| 제외 | **비전 타워(`visual.*`)·`lm_head`** 는 `ignore` → **언어 디코더 선형층만 4bit** |
| oneshot | `sequential_targets=['Qwen3VLTextDecoderLayer']` (레이어 단위 순차 처리, 메모리 절약) |
| 저장 | `save_pretrained(..., save_compressed=True)` |


In [ ]:
# ── AWQ 양자화 함수 ──────────────────────────────────────────
def quantize_awq(model_path: str, save_dir: str, n_calib=N_CALIB, max_seq_len=MAX_SEQ_LEN):
    """model_path(또는 repo id)를 AWQ W4A16으로 양자화해 save_dir에 저장."""
    
    # 1) 모델·프로세서 로드 (dtype='auto' → 체크포인트 native dtype = bf16)
    # Qwen3-VL 멀티모달 모델 로드
    model = Qwen3VLForConditionalGeneration.from_pretrained(model_path, dtype='auto')
    # 이미지+텍스트 전처리 프로세서 로드(토큰화+이미지 변환)
    processor = AutoProcessor.from_pretrained(model_path)

    # 2) ChartQA calibration 서브셋
    # HuggingFaceM4에서 ChartQA 데이터셋의 train split 로드 → 셔플 → n_calib개만 선택
    ds = load_dataset('HuggingFaceM4/ChartQA', split='train').shuffle(seed=42).select(range(n_calib))

    # 3) 전처리: 이미지+실제 query를 멀티모달 입력으로 토큰화
    def preprocess(example):
        # 이미지를 PNG 바이트로 변환
        buffered = BytesIO()
        example['image'].save(buffered, format='PNG')
        # 바이트를 base64 문자열로 인코딩 (모델 입력용)
        b64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
        # data URI 형식으로 변환
        uri = f'data:image;base64,{b64}'
        # 멀티모달 메시지: 이미지 + 질문 텍스트
        messages = [{'role': 'user', 'content': [
            {'type': 'image', 'image': uri},
            {'type': 'text',  'text': example['query']}]}]
        # 채팅 템플릿 적용 (Qwen 포맷 문자열 생성)
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        # 메시지에서 비전 정보(이미지/비디오) 추출
        image_inputs, video_inputs = process_vision_info(messages)
        # 토큰화 + 패딩/자르기 (최대 시퀀스 길이 제한)
        return processor(text=[text], images=image_inputs, videos=video_inputs,
                         padding=False, max_length=max_seq_len, truncation=True)
    # 데이터셋 전체에 전처리 함수 적용, 원본 컬럼 제거
    ds = ds.map(preprocess, remove_columns=ds.column_names)

    # 4) 멀티모달 콜레이터(배치 1개씩 → 텐서화)
    def data_collator(batch):
        # oneshot 호출 시 배치 크기가 1이라고 가정
        assert len(batch) == 1
        # 모든 필드를 PyTorch 텐서로 변환 (GPU 호환성)
        return {k: torch.tensor(v) for k, v in batch[0].items()}

    # 5) 레시피 (Qwen3-VL 공식 AWQ 형태 · 2-modifier):
    #    AWQModifier()가 스케일 탐색 → QuantizationModifier가 W4A16 양자화 수행.
    #    (이 버전의 AWQModifier는 scheme/targets/ignore를 받지 않음 → QuantizationModifier에 지정)
    #    비전 타워(visual.*, model.visual.*)와 lm_head 는 제외 → 언어 디코더만 4bit.
    recipe = [
        # 첫 번째 modifier: 활성값 기반 스케일 탐색 및 스무딩
        AWQModifier(),
        # 두 번째 modifier: 실제 W4A16 양자화 수행
        QuantizationModifier(
            targets="Linear",                                    # 선형층만 양자화 대상
            scheme="W4A16",                                      # 가중치 4bit, 활성 16bit
            ignore=["lm_head", "re:visual.*", "re:model.visual.*"],  # 이 레이어들은 제외(원본 유지)
        ),
    ]

    # 6) oneshot 실행(= calibration + 양자화)
    # calibration 데이터를 통과하며 AWQ 스케일 탐색 후 가중치 양자화
    oneshot(
        model=model, 
        tokenizer=model_path,                          # 토크나이저 for 길이 정보
        dataset=ds,                                    # calibration 데이터셋
        recipe=recipe,                                 # 2-modifier 레시피
        max_seq_length=max_seq_len,                    # 최대 시퀀스 길이
        num_calibration_samples=n_calib,               # calibration 샘플 수
        data_collator=data_collator,                   # 배치 조립 함수
        sequential_targets=['Qwen3VLTextDecoderLayer'],  # 메모리 절약: 디코더 레이어를 순차 처리
    )

    # 7) 압축 저장
    # save_compressed=True → ZSTD 압축하여 디스크 공간 절약
    model.save_pretrained(save_dir, save_compressed=True)
    # 프로세서(토큰화기, 이미지 프로세서)도 함께 저장
    processor.save_pretrained(save_dir)

    # 8) 메모리 해제(다음 모델을 위해 GPU/RAM 정리)
    del model                      # 모델 메모리 해제
    gc.collect()                   # Python 가비지 컬렉션
    torch.cuda.empty_cache()       # GPU 캐시 초기화
    print('저장 완료:', save_dir)
    return save_dir

def merge_lora(base_id, adapter_id, out_dir):
    """base + LoRA 어댑터 → 병합 fp16 단일 모델 저장 (A100·메모리 여유)."""
    
    # (안전) 구버전 torchao가 있으면 PEFT 병합과 충돌 → 그때만 정리(최신/없으면 그대로)
    # torchao < 0.16.0 버전은 PEFT merge와 호환성 문제가 있으므로 제거
    try:
        import importlib.metadata as _md
        from packaging.version import parse as _p
        if _p(_md.version('torchao')) < _p('0.16.0'):
            import sys, subprocess
            # 구버전 torchao 제거
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
            # 이미 로드된 torchao 모듈을 메모리에서 제거
            for _m in [m for m in list(sys.modules) if m == 'torchao' or m.startswith('torchao.')]:
                del sys.modules[_m]
    except Exception:
        pass  # torchao 없거나 버전 확인 불가능 → 그냥 진행
    
    from peft import PeftModel
    
    # base 모델 로드 (fp16/bf16으로 로드 → 메모리 절약)
    base = Qwen3VLForConditionalGeneration.from_pretrained(base_id, dtype=torch.bfloat16)
    # LoRA 어댑터 로드 + base 가중치에 병합(merge_and_unload())
    # merge_and_unload() → 어댑터의 저순위 행렬을 base의 가중치에 더해서 단일 모델로 변환
    merged = PeftModel.from_pretrained(base, adapter_id).merge_and_unload()
    # 병합된 모델을 저장 (safe_serialization=True → safetensors 형식, 더 안전)
    merged.save_pretrained(out_dir, safe_serialization=True)
    # 어댑터 폴더의 프로세서도 함께 저장 (병합본도 같은 프로세서 사용)
    AutoProcessor.from_pretrained(adapter_id).save_pretrained(out_dir)
    
    # 메모리 해제
    del base, merged               # base + merged 모델 메모리 해제
    gc.collect()                   # Python 가비지 컬렉션
    torch.cuda.empty_cache()       # GPU 캐시 초기화
    print('병합 완료:', out_dir)
    return out_dir

print('quantize_awq / merge_lora 정의 완료')

quantize_awq / merge_lora 정의 완료


**▸ 실행 결과 읽기.** `quantize_awq / merge_lora 정의 완료` — 함수만 정의됐고 아직 실행 전입니다. 핵심은 레시피가 **두 modifier**라는 점: `AWQModifier()`가 채널 스케일을 찾고(스무딩), `QuantizationModifier(W4A16)`가 실제 4bit 변환을 합니다. `ignore`에 비전·`lm_head`를 넣어 **언어 디코더만** 양자화합니다.

## 2. 4B — 어댑터 병합 후 AWQ 생성 (Day1 어댑터)

4B는 Day1에서 만든 **LoRA 어댑터**가 있습니다. 양자화는 **단일 모델**을 대상으로 하므로, 먼저 `merge_lora`로 **어댑터를 base 가중치에 흡수**(병합 fp16)한 뒤 그 병합본을 AWQ로 변환합니다. (Colab과 달리 **A100는 메모리 여유**가 있어 병합을 여기서 합니다 — 이것이 Day1은 어댑터만 올리고 Day2에서 병합하는 'A안' 설계입니다.)

In [10]:
# ── 4B: Day1 어댑터를 base에 병합(A100·메모리 여유) → AWQ ──
MERGED_4B_DIR = f'{WORKDIR}/Qwen3-VL-4B-ChartQA-merged'
merge_lora(BASE_4B, ADAPTER_REPO, MERGED_4B_DIR)   # base 4B + 어댑터 → 병합 fp16
AWQ_4B_DIR = f'{WORKDIR}/Qwen3-VL-4B-ChartQA-AWQ'
quantize_awq(MERGED_4B_DIR, AWQ_4B_DIR)            # 병합본을 AWQ W4A16

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

병합 완료: /workspace/VLM_FT_2/Qwen3-VL-4B-ChartQA-merged


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

...

Compressing model: 100%|██████████| 252/252 [00:00<00:00, 947.47it/s]
/workspace/VLM_FT_2/.venv-quant/lib/python3.12/site-packages/transformers/modeling_utils.py:3470: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (50GB default)
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Dispatching model: 100%|██████████| 815/815 [00:00<00:00, 36974.01it/s]


저장 완료: /workspace/VLM_FT_2/Qwen3-VL-4B-ChartQA-AWQ


'/workspace/VLM_FT_2/Qwen3-VL-4B-ChartQA-AWQ'

**▸ 실행 결과 읽기 — AWQ 생성 로그 해독**

이 셀은 두 가지를 차례로 합니다: **① 병합**(`merge_lora`) → **② AWQ 양자화**(`quantize_awq`).

**① 병합 단계**
- `Loading weights … Writing model shards … 병합 완료` — base 4B에 LoRA 어댑터를 흡수해 단일 fp16 모델로 저장했습니다.

**② AWQ 단계 — 로그 메시지의 의미**
- `Inferred SequentialPipeline for AWQModifier` — 메모리 절약을 위해 **디코더 레이어를 하나씩 순차** 처리한다는 뜻입니다.
- **`Preparing cache: 256/256 [01:32]`** — 보정 256개를 모델에 한 번 통과시켜 각 레이어의 **입력 활성값을 캐시**합니다. 이미지까지 forward하므로 가장 오래 걸립니다(8B는 04:15).
- **`(N/37): Calibrating`** — N번째 레이어에서 활성값 통계를 모아 **어느 입력 채널이 중요(salient)한지** 파악합니다. (37 = 디코더 레이어 수 + 경계)
- **`Smoothing: 3/3`** — AWQ의 핵심. 중요한 활성 채널은 **스케일을 낮추고** 대응 **가중치 채널은 스케일을 높여**(출력은 동일) 양자화 손상을 줄입니다. 레이어당 매핑 그룹 수(3)만큼 수행합니다. (첫 레이어는 적용 대상이 없어 `0it`)
- **`(N/37): Propagating`** — 스무딩된 레이어에 보정 데이터를 다시 흘려 **다음 레이어 입력으로 전파**합니다. 그래서 레이어마다 *Calibrating → Smoothing → Propagating* 이 반복됩니다.
- `Inferred DataFreePipeline for QuantizationModifier` — 두 번째 modifier(실제 4bit 변환)는 **데이터가 필요 없습니다**. 위에서 찾은 스케일로 가중치를 4bit로 반올림만 합니다.
- **`Compressing model: 252/252`** — 252개 선형층 가중치를 4bit로 **패킹**합니다.
- `post_process … Optimized model is not saved` (WARNING) — **무해**합니다. `oneshot`에 `output_dir`를 주지 않아 자동 저장만 생략됐고, 우리는 바로 다음 줄 `save_pretrained(save_compressed=True)`로 **직접 저장**합니다.

### (보충) 왜 '활성 채널'이 기준인가 — 미니 수치 예시

**활성 채널**은 선형층에 들어오는 **입력 벡터의 각 차원**이고, 값이 큰 채널일수록 출력에 큰 영향을 줍니다(=salient). 아래 4채널 예시로 스무딩이 왜 효과가 있는지 직접 확인합니다.

- 채널4는 **활성값이 30**으로 크고(salient), 가중치는 0.05로 작지만 **기여(0.05×30 = 1.5)는 큽니다**.
- 그냥 4bit로 양자화하면 작은 0.05가 **격자에서 0으로 뭉개져** 그 기여(1.5)가 통째로 사라집니다 → 출력이 4.2 → 2.7로 크게 틀립니다.
- AWQ는 이 채널의 **가중치를 ×8, 활성을 ÷8**(곱은 그대로 → 출력 불변)로 옮겨 0.05를 0.4로 키웁니다. 이제 4bit 격자에서 **살아남아**(0.386) 오차가 36% → 1.3%로 줄어듭니다.

> 핵심: **무엇을 지킬지(=중요 가중치)를 '활성 채널의 크기'로 판단**하고, 출력을 바꾸지 않으면서 그 가중치를 양자화하기 쉬운 크기로 옮기는 것 — 이것이 **activation-aware**(활성 인지)의 의미입니다. (4bit가 되는 건 가중치, 활성은 16bit로 흐릅니다 = `W4A16`)

In [ ]:
# ── (보충) AWQ 스무딩이 양자화 오차를 줄이는 원리 — 미니 수치 예시 ──
import numpy as np

def quant_4bit(w):
    """per-tensor absmax 대칭 4bit(레벨 -7..7) 양자화"""
    # 가중치 전체에서 최대 절댓값을 기준으로 4bit 스케일 계산
    scale = np.abs(w).max() / 7
    # 스케일로 나눈 뒤 반올림하고, 표현 범위(-7~7)로 자른 후 다시 원래 스케일로 복원
    return np.round(w / scale).clip(-7, 7) * scale

# 입력 4채널 중 채널4가 'salient'(활성값 30)인 예시
# 가중치는 작지만 활성값이 크기 때문에 실제 기여도가 큼
x = np.array([1.0, 1.0, 1.0, 30.0])
w = np.array([0.9, 0.9, 0.9, 0.05])

# 원래 가중치와 입력으로 계산한 정답 출력
y_true = w @ x
print(f'정답 출력 y = {y_true:.2f}  (채널4 기여 = 0.05×30 = 1.5)')

# (A) 스무딩 없이 바로 4bit 양자화
wq = quant_4bit(w)
yA = wq @ x
print(f'\n[A] 스무딩 없음  : 양자화 가중치 {np.round(wq,3).tolist()}')
print(f'    → 채널4 가중치 0.05가 0으로 뭉개짐 → y={yA:.2f}, 오차 {abs(yA-y_true):.2f} ({abs(yA-y_true)/y_true*100:.0f}%)')

# (B) AWQ 스무딩:
# 중요 채널(채널4)의 가중치는 키우고, 활성값은 줄여서 곱은 유지
# 이렇게 하면 양자화 시 작은 가중치가 0으로 사라질 가능성이 줄어듦
s = np.array([1, 1, 1, 8.0])
w2, x2 = w * s, x / s

# 스무딩된 가중치를 4bit로 양자화
wq2 = quant_4bit(w2)
yB = wq2 @ x2
print(f'\n[B] AWQ 스무딩 후: 채널4 가중치 0.05→{w2[3]:.1f}, 활성 30→{x2[3]:.2f}')
print(f'    → 양자화 가중치 {np.round(wq2,3).tolist()} (채널4 살아남음)')
print(f'    → y={yB:.2f}, 오차 {abs(yB-y_true):.2f} ({abs(yB-y_true)/y_true*100:.1f}%)')

# 스무딩 전/후 오차 비교
print(f'\n오차: {abs(yA-y_true):.2f} → {abs(yB-y_true):.2f}  (스무딩이 중요 채널을 지켜 오차 급감)')

정답 출력 y = 4.20  (채널4 기여 = 0.05×30 = 1.5)

[A] 스무딩 없음  : 양자화 가중치 [0.9, 0.9, 0.9, 0.0]
    → 채널4 가중치 0.05가 0으로 뭉개짐 → y=2.70, 오차 1.50 (36%)

[B] AWQ 스무딩 후: 채널4 가중치 0.05→0.4, 활성 30→3.75
    → 양자화 가중치 [0.9, 0.9, 0.9, 0.386] (채널4 살아남음)
    → y=4.15, 오차 0.05 (1.3%)

오차: 1.50 → 0.05  (스무딩이 중요 채널을 지켜 오차 급감)


## 3. 8B AWQ 생성 (베이스 모델)

8B는 **별도 어댑터가 없으므로**(파인튜닝은 4B만 했습니다) **base 8B를 그대로** AWQ로 변환합니다. 같은 `quantize_awq` 함수를 재사용합니다 — 4B와 동일한 파이프라인이 한 번 더 도는 것뿐입니다.

In [11]:
# ── 8B AWQ ───────────────────────────────────────────────────
AWQ_8B_DIR = f'{WORKDIR}/Qwen3-VL-8B-AWQ'
quantize_awq(BASE_8B, AWQ_8B_DIR)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

2026-06-27T11:53:56.3894 | reset | INFO - Compression lifecycle reset
2026-06-27T11:53:56.3974 | from_modifiers | INFO - Creating recipe from modifiers
2026-06-27T11:53:56.3979 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...
2026-06-27T11:53:56.3982 | get_layer_mappings_from_model | INFO - Architecture Qwen3VLForConditionalGeneration not found in mappings. Using default mappings: [AWQMapping(smooth_layer='re:.*input_layernorm$', balance_layers=['re:.*q_proj$', 're:.*k_proj$', 're:.*v_proj$'], activation_hook_target=None), AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None), AWQMapping(smooth_layer='re:.*post_attention_layernorm$', balance_layers=['re:.*gate_proj$', 're:.*up_proj$'], activation_hook_target=None), AWQMapping(smooth_layer='re:.*up_proj$', balance_layers=['re:.*down_proj$'], activation_hook_target=None)]
2026-06-27T11:53:56.6640 | initialize | INFO - Compression lifecycle initialized for 2

(1/37): Calibrating: 100%|██████████| 256/256 [00:09<00:00, 27.89it/s]
Smoothing: 0it [00:00, ?it/s]
(37/37): Propagating: 100%|██████████| 256/256 [00:00<00:00, 506.49it/s]

...

Compressing model: 100%|██████████| 252/252 [00:00<00:00, 690.01it/s]
/workspace/VLM_FT_2/.venv-quant/lib/python3.12/site-packages/transformers/modeling_utils.py:3470: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (50GB default)
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Dispatching model: 100%|██████████| 845/845 [00:00<00:00, 32734.71it/s]


저장 완료: /workspace/VLM_FT_2/Qwen3-VL-8B-AWQ


'/workspace/VLM_FT_2/Qwen3-VL-8B-AWQ'

**▸ 실행 결과 읽기.** 4B와 **완전히 같은 단계**(Preparing cache → Calibrating → Smoothing → Propagating → Compressing)가 반복됩니다. 다만 모델이 커서 **더 느립니다** 

## 4. 모델 크기 비교 — 얼마나 줄었나

원본(bf16, 파라미터×2바이트 추정)과 AWQ 디렉터리 크기를 비교합니다. **4bit인데 왜 정확히 4배가 안 줄까?** 를 결과에서 직접 확인하세요(아래 해석 참고).

In [ ]:
# ── 디렉터리 크기 측정 ──────────────────────────────────────
def dir_size_gb(path: str) -> float:
    """디렉터리의 모든 파일 크기를 합산해 GB 단위로 반환"""
    total = 0
    # os.walk()로 디렉터리 트리를 재귀적으로 순회
    # root: 현재 디렉터리, _: 서브디렉터리(사용 안 함), files: 파일 리스트
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)  # 파일의 절대 경로
            if os.path.isfile(fp):
                total += os.path.getsize(fp)  # 파일 크기(바이트)를 누적
    return total / (1024 ** 3)  # 총합을 GB로 변환 (1GB = 1024^3 바이트)

# ── 원본 bf16 크기 추정 ──
# Qwen3-VL 공개 스펙 기준 파라미터 수 × 2바이트(bf16 float = 16bit = 2바이트)
orig_4b = 4_437_800_000 * 2 / 1024**3   # 약 8.3 GB (bf16)
orig_8b = 8_767_100_000 * 2 / 1024**3   # 약 16.3 GB (bf16)

# ── AWQ 저장된 모델의 실제 디스크 크기 측정 ──
# save_compressed=True로 저장된 폴더를 순회해 실제 파일 크기 합산
awq_4b_gb = dir_size_gb(AWQ_4B_DIR)
awq_8b_gb = dir_size_gb(AWQ_8B_DIR)

# ── 결과 표 출력 ──
# 모델명 | 원본(bf16) | AWQ(4bit) | 압축률(배수)
print(f'{"모델":<10}{"원본(bf16)":>14}{"AWQ(4bit)":>14}{"압축률":>10}')
print('-' * 50)
# 4B: 원본 크기 / AWQ 크기 = 압축률(x배)
print(f'{"4B":<10}{orig_4b:>11.1f} GB{awq_4b_gb:>11.2f} GB{orig_4b/awq_4b_gb:>9.1f}x')
# 8B: 원본 크기 / AWQ 크기 = 압축률(x배)
print(f'{"8B":<10}{orig_8b:>11.1f} GB{awq_8b_gb:>11.2f} GB{orig_8b/awq_8b_gb:>9.1f}x')

모델              원본(bf16)     AWQ(4bit)       압축률
--------------------------------------------------
4B                8.3 GB       3.98 GB      2.1x
8B               16.3 GB       6.74 GB      2.4x


**▸ 실행 결과 읽기.**

| 모델 | 원본(bf16) | AWQ(4bit) | 압축률 |
|---|---|---|---|
| 4B | 8.3 GB | 3.98 GB | **2.1×** |
| 8B | 16.3 GB | 6.74 GB | **2.4×** |

**왜 4배가 아니라 ~2배인가?** 16bit→4bit는 이론상 4배지만, 실제로는 **언어 디코더의 선형층만** 4bit로 줄었습니다. 다음은 그대로 남아 전체 절감을 희석합니다:
- **비전 타워(ViT)** 와 **`lm_head`, 임베딩** — `ignore`로 제외해 **bf16 유지**(지각·출력 품질 보존). VLM은 비전 비중이 커서 영향이 큽니다.
- **4bit 패킹 부가정보** — 그룹별 **scale/zero-point** 등이 함께 저장돼 순수 4bit보다 약간 큽니다.

그래서 "전부 4bit"가 아니라 "**중요 부분은 지키고 언어층만 4bit**" 한 결과가 ~2배입니다. 8B가 4B보다 압축률이 높은 건 언어 디코더 비중이 더 크기 때문입니다.

## 5. HuggingFace Hub에 업로드

만든 두 AWQ 모델을 Hub에 올려두면:
- **보존**: RunPod pod이 사라져도 결과물이 남습니다.
- **재사용**: Day2-5 서빙이나 다른 환경에서 **repo id만으로 바로 다운로드**해 씁니다.
- **공유**: 동료·교육생이 같은 모델을 받아 실습할 수 있습니다.

**repo 네이밍**(어댑터 repo와 같은 계정):
- `<your-username>/Qwen3-VL-4B-ChartQA-AWQ`
- `<your-username>/Qwen3-VL-8B-AWQ`

> 🔑 업로드에는 **write 권한 토큰**이 필요합니다(HF Settings → Access Tokens에서 *write* 발급 → `.env`의 `HF_TOKEN`). `PRIVATE=False`면 **공개 repo**가 됩니다. 베이스 모델이 apache-2.0이라 AWQ 산출물도 같은 라이선스로 표기합니다.

In [ ]:
# ── 두 AWQ 모델을 HuggingFace Hub에 업로드 ──────────────────────
from huggingface_hub import HfApi, create_repo

HF_USER = '<your-username>'     # ← 본인 HF 계정으로 변경
PRIVATE = True                  # True면 비공개. 
AWQ_4B_DIR = f'{WORKDIR}/Qwen3-VL-4B-ChartQA-AWQ'
AWQ_8B_DIR = f'{WORKDIR}/Qwen3-VL-8B-AWQ'

# 업로드 대상: {repo_id: 로컬 폴더}
uploads = {
    f'{HF_USER}/Qwen3-VL-4B-ChartQA-AWQ': (AWQ_4B_DIR, BASE_4B),
    f'{HF_USER}/Qwen3-VL-8B-AWQ':         (AWQ_8B_DIR, BASE_8B),
}

def make_model_card(repo_id, base_model):
    """간단한 모델 카드(README.md) 생성 — YAML 메타 + 사용법."""
    return f'''---
license: apache-2.0
base_model: {base_model}
tags:
- qwen3-vl
- awq
- compressed-tensors
- vllm
- image-text-to-text
pipeline_tag: image-text-to-text
---

# {repo_id.split('/')[-1]}

`{base_model}` 를 **AWQ(W4A16, 4bit)** 로 양자화한 모델입니다(llmcompressor + compressed-tensors).
언어 디코더 선형층만 4bit로 변환하고 **비전 타워와 `lm_head`는 보존**했습니다.

## vLLM 서빙
```bash
vllm serve {repo_id} --served-model-name model \\
  --max-model-len 4096 --gpu-memory-utilization 0.9 --enforce-eager --port 8000
```

## transformers 로드
```python
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
model = Qwen3VLForConditionalGeneration.from_pretrained("{repo_id}", dtype="auto", device_map="cuda")
processor = AutoProcessor.from_pretrained("{repo_id}")
```
'''

api = HfApi(token=os.environ['HF_TOKEN'])
for repo_id, (local_dir, base_model) in uploads.items():
    # 모델 카드가 없으면 생성
    card = os.path.join(local_dir, 'README.md')
    if not os.path.exists(card):
        with open(card, 'w', encoding='utf-8') as f:
            f.write(make_model_card(repo_id, base_model))
    # repo 생성(이미 있으면 통과) 후 폴더 통째로 업로드
    create_repo(repo_id, repo_type='model', private=PRIVATE,
                exist_ok=True, token=os.environ['HF_TOKEN'])
    print(f'업로드 중: {local_dir}  →  {repo_id} ...')
    api.upload_folder(folder_path=local_dir, repo_id=repo_id, repo_type='model')
    print(f'  ✓ https://huggingface.co/{repo_id}')
print('업로드 완료')

업로드 중: /workspace/VLM_FT_2/Qwen3-VL-4B-ChartQA-AWQ  →  nugunaai/Qwen3-VL-4B-ChartQA-AWQ ...
  ✓ https://huggingface.co/nugunaai/Qwen3-VL-4B-ChartQA-AWQ
업로드 중: /workspace/VLM_FT_2/Qwen3-VL-8B-AWQ  →  nugunaai/Qwen3-VL-8B-AWQ ...
  ✓ https://huggingface.co/nugunaai/Qwen3-VL-8B-AWQ
업로드 완료


**▸ 실행 결과 읽기.** 각 모델마다 `업로드 중 … → ✓ https://huggingface.co/...` 가 찍히면 성공입니다. 폴더 전체(safetensors 가중치 + config + 토크나이저 + 방금 만든 README)가 올라갑니다. 8B(≈6.7GB)는 네트워크에 따라 수 분 걸릴 수 있습니다. 업로드 후 그 URL을 열면 모델 카드와 파일 목록이 보입니다.

> 권한 오류(`401`/`403`)가 나면 토큰이 **read 전용**일 때입니다 — **write 토큰**으로 `.env`를 갱신하세요. 같은 repo에 다시 업로드하면 변경된 파일만 갱신됩니다(버전 관리).

## 6. 다운로드해서 사용하기

업로드한 모델은 **repo id만으로** 어디서든 받아 쓸 수 있습니다. 세 가지 방법:

**① vLLM 서빙 (Day2-5와 동일, 로컬 경로 대신 repo id)**
```bash
vllm serve <your-username>/Qwen3-VL-8B-AWQ \
  --served-model-name model --max-model-len 4096 \
  --gpu-memory-utilization 0.9 --enforce-eager --port 8000
```
vLLM이 Hub에서 자동으로 받아 서빙합니다(최초 1회 다운로드 후 캐시).

**② transformers로 바로 로드** — `from_pretrained`에 repo id를 그대로 전달(아래 셀).

**③ 로컬 폴더로 내려받기** — `snapshot_download`로 특정 경로에 저장(오프라인/재현용, 아래 셀).

> 비공개(`PRIVATE=True`) repo라면 다운로드 측에도 `HF_TOKEN`(read 이상)이 필요합니다. 공개 repo면 토큰 없이 받을 수 있습니다.

In [ ]:
# ── (참고) 업로드한 AWQ 모델을 받아서 쓰기 — 필요할 때 실행 ──────────
HF_USER = '<your-username>'                   # 본인 HF 계정명으로 변경
REPO = f'{HF_USER}/Qwen3-VL-8B-AWQ'           # 예: 8B AWQ 모델 repo id

# ── 방법 ③: 로컬 폴더로 내려받기 (오프라인 재현/저장용) ──
# snapshot_download: HF Hub에서 모델 전체를 지정한 폴더에 다운로드
# repo_id: 모델 repo 경로
# local_dir: 로컬 저장 경로 (처음이면 생성, 이미 있으면 갱신)
from huggingface_hub import snapshot_download
local = snapshot_download(repo_id=REPO, local_dir=f'/workspace/dl/{REPO.split("/")[-1]}')
print('다운로드 위치:', local)  # 다운로드 완료 후 저장된 경로 출력

# ── 방법 ②: transformers로 바로 로드 (자동 다운로드 + 메모리 상 로드) ──
# from_pretrained(repo_id, ...) 호출 시:
# - Hub에서 모델 파일 자동 다운로드(~/.cache/huggingface/hub에 캐시)
# - 가중치를 GPU 메모리에 올림 (dtype='auto' = 체크포인트의 원래 dtype 유지)
# - device_map='cuda' = 모든 레이어를 GPU 0번에 배치
# - .eval() = 추론 모드 (드롭아웃 등 비활성화)
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
model = Qwen3VLForConditionalGeneration.from_pretrained(REPO, dtype='auto', device_map='cuda').eval()
processor = AutoProcessor.from_pretrained(REPO)  # 토크나이저 + 이미지 프로세서 로드
print('로드 완료:', model.config.model_type)  # 로드된 모델의 타입(qwen3_vl) 출력

**▸ 실행 결과 읽기.** `다운로드 위치: …` 는 `snapshot_download`가 저장한 로컬 경로, `로드 완료: qwen3_vl` 는 transformers가 AWQ 모델을 인식해 GPU에 올렸다는 뜻입니다(로딩 중 [4.]에서 본 *Decompressing model* 로그가 다시 보입니다). 이 셀은 실제 다운로드(6.7GB)를 일으키므로 **필요할 때만** 실행하세요.

## 7. 정리 + 다음 교시 예고

- `AWQModifier` + `QuantizationModifier(W4A16)` 로 **4B·8B를 AWQ 4bit로 변환**했습니다(중요 채널 스무딩 → 4bit 패킹).
- 비전 타워·`lm_head`를 **`ignore`** 해 지각·출력 품질을 보존했습니다.
- 디스크 크기가 약 **2배(4B 2.1× · 8B 2.4×)** 줄었습니다 — 전부 4bit가 아니라 **언어 디코더 선형층만** 줄였기 때문입니다.
- 두 모델을 **HuggingFace Hub에 업로드**하고, **repo id로 다운로드/서빙**하는 법을 익혔습니다.

**산출물(로컬)**: `Qwen3-VL-4B-ChartQA-AWQ`, `Qwen3-VL-8B-AWQ`
**산출물(Hub)**: `nugunaai/Qwen3-VL-4B-ChartQA-AWQ`, `nugunaai/Qwen3-VL-8B-AWQ`

### 다음 교시 — Day2-4 · 양자화 평가
test셋에서 **Relaxed Accuracy 4종**(원본/AWQ × 4B/8B)을 측정하고, 크기·VRAM·정확도를 한 표로 비교합니다. 손실이 큰 케이스(세밀 차트·숫자)도 분석합니다.

> ✅ **체크포인트**: ① AWQ 레시피의 두 modifier 역할을 안다 ② 왜 비전을 ignore하는지 안다 ③ 로그의 Calibrating/Smoothing/Propagating 의미를 안다 ④ 4bit인데 ~2배만 주는 이유를 안다 ⑤ Hub 업로드·다운로드 흐름을 안다.

## 경량화  
- 사이즈를 줄이자

## fine tunning
- 성능을 높이자

fine tunning한 것을 경량화하는 것이 Good!!!

## 실습
- 4B + LoRA Adapter(fine tunning) => merged model(4B)=>AWQ
- 8B => AWQ 
- 둘 중 위의 것이 Good